<div style="padding:2.2rem;border-radius:18px;background:linear-gradient(135deg,#3f1d68,#0f766e);color:white;">
<p style="font-size:1.05rem;letter-spacing:.12em;text-transform:uppercase;opacity:.85;">D082 · Beginner Architecture</p>
<h1 style="font-size:3.1rem;margin:.3rem 0;">Data Architecture Overview</h1>
<p style="font-size:1.4rem;max-width:900px;">Operational systems, analytical systems, and the movement of data between them.</p>
<hr style="border:0;border-top:1px solid rgba(255,255,255,.35);margin:2rem 0;">
<p>Conceptual · Vendor-neutral · Approx. 2 hours</p>
</div>


# Learning outcomes

By the end, you should be able to:

- Explain the purpose of operational and analytical systems
- Describe how a relational OLTP system supports transactions
- Explain concurrency, locks, and their trade-offs
- Recognize common OLTP scaling approaches
- Explain partitioning and sharding conceptually
- Describe a data warehouse and MPP architecture
- Explain why OLTP data is moved to OLAP systems
- Compare batch, incremental, and streaming movement
- Read a simple end-to-end data architecture


# Running example: **ShopNow**

ShopNow must support two very different needs:

### Run the business

- Place orders
- Reserve inventory
- Record payments
- Update shipment status

### Understand the business

- Compare monthly revenue
- Identify popular products
- Measure delivery performance
- Forecast demand

One architecture rarely serves both needs equally well.


# The big picture

```text
CUSTOMERS & STAFF
       ↓
┌─────────────────────┐
│ OPERATIONAL SYSTEMS │  run current business processes
└──────────┬──────────┘
           │ data movement
           ▼
┌─────────────────────┐
│ ANALYTICAL SYSTEMS  │  preserve, combine, and analyze history
└──────────┬──────────┘
           ▼
     decisions & action
```

The architecture separates workloads while connecting their data.


# Architecture is about trade-offs

An architecture answers:

- Which responsibilities belong together?
- Which responsibilities should be separated?
- Where is data created and stored?
- How does data move?
- What happens under high demand?
- What happens when a component fails?
- Who consumes each result?

> Architecture is the deliberate arrangement of responsibilities and boundaries—not a diagram of product logos.


# Part 1 — Operational systems

<div style="padding:1.5rem;border-left:8px solid #0f766e;background:#ccfbf1;">
<h2 style="margin-top:0;">Systems that run the organization now</h2>
<p style="font-size:1.2rem;">They record individual business events and maintain current state safely and quickly.</p>
</div>


# What is an operational system?

An operational system supports a business process:

- E-commerce checkout
- Airline reservation
- Bank transfer
- Hospital registration
- University enrollment
- Warehouse inventory update

Typical requests affect a small number of records:

```text
Create order O-1057
Reserve 2 units of P-18
Record payment success
Change order status to “Packed”
```


# OLTP

**Online Transaction Processing (OLTP)** describes systems optimized for many short, concurrent transactions.

Typical characteristics:

- Many users and applications
- Small reads and writes
- Millisecond-to-second response expectations
- Frequent changes to current state
- Strong correctness requirements
- High availability

Examples of database products include PostgreSQL, MySQL, SQL Server, and Oracle—but our focus is the architectural role, not the vendor.


# Relational databases

A relational database organizes data into tables connected by relationships.

```text
CUSTOMERS  1 ─────── many  ORDERS
PRODUCTS   1 ─────── many  ORDER_ITEMS
ORDERS     1 ─────── many  ORDER_ITEMS
```

Useful properties:

- Explicit structure
- Keys and relationships
- Rules that protect valid data
- Transactions
- Flexible queries

Relational design is a strong fit for many operational systems.


# A simplified operational model

| CUSTOMERS | ORDERS | ORDER_ITEMS | PRODUCTS |
|---|---|---|---|
| customer_id | order_id | order_id | product_id |
| name | customer_id | product_id | name |
| address | ordered_at | quantity | price |
| contact | status | item_price | stock |

The structure avoids repeating the same customer and product details in every order.

That helps operational updates remain consistent.


# What is a transaction?

A **transaction** is a unit of work that must be treated as one logical operation.

Placing an order may require:

1. Create the order
2. Add order items
3. Reduce available inventory
4. Record payment state

If a critical step fails, partial work may need to be undone.

> The business action is “place order,” even if several records must change.


# Transaction guarantees

Transactions aim to provide:

- **Atomicity:** all required changes succeed or none do
- **Consistency:** rules remain satisfied
- **Isolation:** concurrent work does not interfere incorrectly
- **Durability:** committed work survives failure

These are commonly remembered as **ACID**.

The guarantees protect business correctness, but stronger coordination can reduce concurrency and increase cost.


# Example: the last item

Only one headset remains.

```text
Customer A reads stock = 1
Customer B reads stock = 1
Customer A places order
Customer B places order
```

Without coordination, both may believe they bought the last unit.

The system must decide how concurrent operations observe and modify shared data.


# Concurrency

**Concurrency** means multiple operations make progress during overlapping time.

It improves throughput:

- Many customers can shop at once
- Many employees can update different records
- Multiple services can use the database

But concurrent operations can conflict when they read or change the same data.

```text
More concurrency → more work completed
Shared data      → greater need for coordination
```


# Common concurrency problems

| Problem | Simple description |
|---|---|
| Lost update | One change silently replaces another |
| Dirty read | Uncommitted data is observed |
| Non-repeatable read | A value changes during one transaction |
| Phantom | A repeated search finds a changed set of rows |

Different isolation choices prevent different problems.

The architectural trade-off is often:

> stronger isolation and simpler reasoning **versus** greater concurrency and throughput.


# Why locks exist

A **lock** coordinates access to shared data.

Conceptually:

```text
Transaction A: acquire lock → change inventory → commit → release
Transaction B:              waits             → continue
```

Locks help prevent unsafe interference.

They do not make work faster; they make concurrent work safer by controlling when it may proceed.


# Shared and exclusive intent

At a high level:

- A **shared/read lock** allows compatible readers
- An **exclusive/write lock** prevents conflicting access while data changes

```text
many readers        ✓ often compatible
reader + writer     ? may require coordination
writer + writer     ✗ conflicting on same data
```

Exact behavior varies by database and isolation method. The architectural idea is controlled access to shared state.


# Row-level locks

A row-level lock protects a small unit of data.

```text
Inventory rows
P-18  ← locked by transaction A
P-19  ← transaction B can update
P-20  ← transaction C can update
```

**Advantages**

- High concurrency when users touch different rows
- Limited blocking

**Trade-offs**

- Many locks may need tracking
- Hot rows still become bottlenecks
- Complex operations may lock many rows


# Table-level locks

A table-level lock protects an entire table.

```text
┌──────── INVENTORY TABLE ────────┐
│ P-18 │ P-19 │ P-20 │ ...       │  ← one broad lock
└─────────────────────────────────┘
```

**Advantages**

- Simpler coordination
- Useful for operations affecting most of a table

**Trade-offs**

- Unrelated operations may wait
- Concurrency can fall sharply
- Long operations have a large impact


# Database-level locks

A database-level lock or equivalent broad exclusive state affects most or all activity in a database.

Possible uses include certain maintenance, administrative, or structural operations.

**Advantages**

- Strong, simple protection for rare global changes

**Trade-offs**

- Very low concurrency
- Large operational impact
- Potential application downtime

> Lock names and exact scopes differ by database; the principle is that broader protection creates broader blocking.


# Lock scope trade-off

| Lock scope | Coordination overhead | Possible concurrency | Blocking impact |
|---|---:|---:|---:|
| Row | Higher | Higher | Narrow |
| Table | Medium | Lower | Broad |
| Database | Lower conceptual complexity | Very low | Very broad |

Smaller scope is not always automatically better.

The appropriate scope depends on how much data an operation touches and how long it runs.


# Lock duration matters

Even a row lock is harmful if held too long.

```text
BEGIN TRANSACTION
  update inventory
  call external payment service  ← slow wait
  write audit record
COMMIT
```

While the transaction waits, other work may block.

Good architecture keeps transactions focused and avoids unnecessary work while shared resources are locked.


# Blocking and lock queues

```text
Transaction A holds lock ─────────────►
Transaction B       waits ────────────► runs
Transaction C            waits ───────► runs
```

One slow transaction can create a queue.

Effects:

- Increased response time
- Reduced throughput
- Timeouts and retries
- Additional load from retries
- Poor user experience

Contention is often concentrated around a small number of “hot” records.


# Deadlocks

A deadlock occurs when transactions wait on each other:

```text
Transaction A holds Row 1; waits for Row 2
Transaction B holds Row 2; waits for Row 1
```

Neither can proceed without intervention.

Databases usually detect the cycle and cancel one transaction.

Architectural lessons:

- Access shared resources in a consistent order
- Keep transactions short
- Handle retries safely


# Concurrency is a workload property

Two systems with the same data volume can behave very differently:

```text
System A: 1 million rows, 10 users
System B: 1 million rows, 10,000 simultaneous users
```

Important questions:

- How many operations occur per second?
- What percentage are reads versus writes?
- Do many operations touch the same records?
- How long do transactions last?
- What response time is required?


# OLTP strengths

- Fast individual operations
- Strong transaction guarantees
- Consistent current state
- Mature relational modeling
- Efficient indexed lookup
- Clear application ownership
- Good support for concurrent business activity

OLTP is excellent at:

> “Find this customer’s current order and update its status safely.”


# OLTP trade-offs

- Long analytical scans compete with transactions
- History may be overwritten
- Data is separated by application
- Complex joins can become expensive at scale
- One machine has practical limits
- Strong coordination can restrict horizontal scaling
- Schema is optimized for updates, not business summaries

OLTP is less suited to:

> “Compare five years of orders across all regions and product categories.”


# Why OLTP systems face scaling pressure

Growth brings:

- More users
- More transactions
- Larger tables and indexes
- More simultaneous reads and writes
- Higher availability expectations
- Hot products, accounts, or inventory rows
- Analytical requests from the business

Scaling is not only about storage size.

It includes **throughput, latency, concurrency, coordination, and recovery**.


# First response: make one machine larger

**Vertical scaling** adds resources to one database server:

```text
more CPU + more memory + faster storage
```

**Advantages**

- Simple application architecture
- Transactions remain local
- Minimal data-distribution complexity

**Trade-offs**

- Hardware has limits
- Larger machines can be expensive
- Failure impact remains concentrated
- Upgrades may require disruption


# Read replicas

A primary database accepts changes; replicas copy those changes and serve selected reads.

```text
                   ┌─► read replica
writes ─► primary ─┼─► read replica
                   └─► read replica
```

**Advantages**

- Distributes read load
- Supports availability and reporting copies

**Trade-offs**

- Replicas can lag behind
- Writes still concentrate on the primary
- Applications must tolerate or route around stale reads


# Partitioning

**Partitioning** divides a large logical table into smaller physical sections.

```text
ORDERS
├── 2024 orders
├── 2025 orders
└── 2026 orders
```

Possible partition keys:

- Date
- Region
- Identifier range
- Category

Partitioning can improve manageability and allow queries to avoid irrelevant sections.


# Partitioning trade-offs

**Advantages**

- Smaller sections to scan or maintain
- Easier lifecycle management
- Some work can run independently

**Trade-offs**

- Poor keys create uneven partitions
- Queries crossing partitions still require work
- Unique rules and relationships may become harder
- Partitioning does not automatically distribute load across machines

Partitioning organizes data; it is not always the same as sharding.


# Sharding

**Sharding** distributes data across independent database nodes.

```text
customer_id A–H ─► Shard 1
customer_id I–P ─► Shard 2
customer_id Q–Z ─► Shard 3
```

Each shard owns part of the data and can process work independently.

The aim is to scale storage and transaction throughput beyond one machine.


# Choosing a shard key

A shard key decides where each record belongs.

A useful key should:

- Distribute data evenly
- Distribute request load evenly
- Keep commonly related data together
- Be stable
- Allow efficient routing

Poor example:

> Shard by country when 80% of customers are in one country.

That creates a hot shard.


# Sharding advantages

- Storage can grow across machines
- Transaction load can be distributed
- Independent shards may fail or scale separately
- Requests confined to one shard can be efficient

```text
one overloaded database
          ↓
multiple independently working shards
```

Sharding can unlock scale—but only when work can be divided effectively.


# Sharding trade-offs

- Application or routing becomes more complex
- Cross-shard joins are harder
- Cross-shard transactions require coordination
- Rebalancing data is difficult
- Hot shards can remain
- Global uniqueness needs design
- Backup, recovery, and monitoring multiply

> Sharding exchanges the limits of one machine for the complexity of distributed coordination.


# Scaling choices in context

| Approach | Primarily helps | Main trade-off |
|---|---|---|
| Vertical scaling | Overall single-node capacity | Cost and finite limit |
| Read replicas | Read throughput | Lag; not write scaling |
| Partitioning | Manageability and selective access | Key/design complexity |
| Sharding | Storage and write distribution | Distributed complexity |

These approaches may be combined.

Start with the simplest architecture that satisfies the real workload.


# Discussion: flash sale

ShopNow launches a limited product:

- 100 units
- 100,000 interested customers
- Most requests target the same inventory record
- Customers expect immediate confirmation

Questions:

1. Why does adding read replicas not solve the write conflict?
2. What becomes the “hot” resource?
3. How might long transactions worsen the problem?
4. What correctness guarantee must not be lost?

**Discuss for 4 minutes.**


# Part 2 — Analytical systems

<div style="padding:1.5rem;border-left:8px solid #7c3aed;background:#ede9fe;">
<h2 style="margin-top:0;">Systems designed to understand patterns across data</h2>
<p style="font-size:1.2rem;">They preserve history, integrate domains, and process large analytical queries.</p>
</div>


# OLAP

**Online Analytical Processing (OLAP)** supports complex analysis over many records.

Examples:

- Revenue by region over five years
- Average delivery delay by warehouse
- Customer retention by acquisition channel
- Product demand by season

Typical characteristics:

- Large reads and aggregations
- Historical data
- Data from multiple sources
- Fewer writes than OLTP
- Structure designed for analytical questions


# OLTP and OLAP side by side

| | OLTP | OLAP |
|---|---|---|
| Purpose | Run processes | Analyze patterns |
| Data scope | Current application state | Integrated history |
| Typical work | One/few records | Millions/billions of records |
| Workload | Many short reads/writes | Fewer, long complex reads |
| Priority | Transaction correctness and latency | Scan and aggregation performance |
| Users | Applications and operators | Analysts and decision-makers |


# What is a data warehouse?

A **data warehouse** is an analytical system containing integrated, structured, historical data.

```text
Orders ──┐
Products ┼─► standardize + integrate ─► DATA WAREHOUSE
Stores ──┤                                  ↓
Finance ─┘                          reports and analysis
```

It creates a consistent analytical view across operational boundaries.


# Warehouse strengths

- Historical analysis
- Integrated data from many systems
- Consistent business definitions
- Efficient aggregations
- Separation from operational workload
- Repeatable reporting
- Structures understandable to analysts

Example:

> “Net revenue” can be defined once using orders, refunds, discounts, and taxes rather than reinterpreted in every report.


# Warehouse trade-offs

- Data may not be immediately current
- Preparation and modeling require effort
- Source changes must be managed
- Incorrect definitions can spread widely
- More copies require governance and security
- Some detail may be transformed or summarized

A warehouse improves analytical use; it does not automatically guarantee correct meaning.


# Dimensional organization

Analytical data is often organized around a measurable business event:

```text
                 DATE
                   │
CUSTOMER ─────── SALE ─────── PRODUCT
                   │
                 STORE
```

- **Fact:** the event—one sale or order item
- **Measures:** quantity, revenue, discount
- **Dimensions:** date, customer, product, store

This shape supports natural business questions.


# Grain: what does one fact row represent?

Possible grains:

- One order
- One product within an order
- One shipment
- One daily product-store summary

Example:

```text
Order O-1057 contains 3 products
Order-grain dataset      → 1 row
Order-item-grain dataset → 3 rows
```

Mixing grains can duplicate totals. Always state what one row means.


# Analytical storage behavior

Analytical systems are designed to:

- Read selected columns across many rows
- Filter large histories
- Group by dimensions
- Calculate aggregates
- Run independent analytical queries concurrently
- Add data in large or continuous loads

They often favor scan efficiency and parallel work over frequent single-row updates.


# MPP architecture

**Massively Parallel Processing (MPP)** distributes analytical work across multiple compute nodes.

```text
                 query
                   ↓
             coordinator
          ┌────────┼────────┐
          ▼        ▼        ▼
       node 1   node 2   node 3
       data A   data B   data C
          └────────┼────────┘
                   ↓
             combined result
```

Large work is divided, processed in parallel, and merged.


# How MPP helps

Suppose a warehouse contains 3 billion sales rows.

```text
Node 1 scans one portion
Node 2 scans another portion
Node 3 scans another portion
...
```

Each node calculates a partial result. The system combines them.

**Advantages**

- Large scans complete faster
- Capacity can expand across nodes
- Analytical workloads benefit from parallelism


# MPP trade-offs

- Data distribution affects performance
- Uneven data causes some nodes to do more work
- Moving data between nodes is expensive
- Large joins may require redistribution
- More nodes add operational coordination
- Small single-record transactions may not benefit

MPP is powerful when work can be split and local processing dominates communication.


# Data distribution and skew

Data must be distributed across MPP nodes.

Good distribution:

```text
Node 1: 34%   Node 2: 33%   Node 3: 33%
```

Skewed distribution:

```text
Node 1: 80%   Node 2: 10%   Node 3: 10%
```

The query waits for the busiest node.

This resembles the hot-shard problem: parallel architecture still depends on balanced work.


# MPP is different from OLTP sharding

Both distribute data, but their primary goals differ:

| OLTP sharding | Analytical MPP |
|---|---|
| Distribute transactions | Parallelize large analytical queries |
| Route small operations to a shard | Coordinate work across many nodes |
| Avoid cross-shard work when possible | Combine partial results routinely |
| Optimize low-latency changes | Optimize scans and aggregations |

Similar mechanism; different workload and coordination pattern.


# Analytical product orientation

Examples of analytical database and warehouse products include Teradata, Snowflake, ClickHouse, Redshift, and BigQuery.

The names are less important than the shared architectural purpose:

> Store and process substantial historical data for analysis.

Products differ in storage, execution, management, and deployment. We are not comparing them in this lesson.


# Part 3 — Data movement

<div style="padding:1.5rem;border-left:8px solid #ea580c;background:#ffedd5;">
<h2 style="margin-top:0;">Connecting systems with different responsibilities</h2>
<p style="font-size:1.2rem;">Movement carries operational evidence into analytical environments without making them the same system.</p>
</div>


# Why move data from OLTP to OLAP?

1. Protect operational response times
2. Preserve history
3. Combine multiple operational systems
4. Apply shared business definitions
5. Reshape data for analytical queries
6. Scale large scans independently
7. Provide governed access to consumers

```text
OLTP: “What is order O-1057 doing now?”
OLAP: “How did order behavior change this year?”
```


# The workload conflict

Without separation:

```text
customer checkout ─┐
inventory update ──┼─► same database ◄─ five-year revenue scan
payment update ────┘
```

The analytical scan can consume CPU, memory, storage bandwidth, and locks needed by customers.

With separation:

```text
operational database ── copy ──► analytical system
fast transactions                   large analysis
```


# Movement is more than copying

Data movement may also:

- Select required records and fields
- Convert formats and types
- Standardize units and timestamps
- Detect duplicates
- Validate required values
- Join reference information
- Record arrival and processing time
- Protect or mask sensitive fields

Every transformation should preserve traceability to its source.


# Batch movement

Batch moves a bounded collection on a schedule.

```text
OLTP orders at midnight
        ↓ daily extraction
        ↓ validate and prepare
warehouse ready by 06:00
```

**Advantages**

- Simpler operational model
- Efficient for large groups
- Clear processing boundaries

**Trade-offs**

- Data is stale between runs
- Large batches create load spikes
- Failure may delay an entire delivery


# Full versus incremental batch

### Full movement

Copy the complete dataset each time.

- Simple to understand
- Expensive as data grows

### Incremental movement

Copy only new or changed records.

- Less data and lower source load
- Must reliably identify changes
- Updates and deletions need careful handling

```text
Yesterday: all orders  |  Today: only changes since yesterday
```


# Change data capture

**Change Data Capture (CDC)** identifies inserts, updates, and deletes from a source.

```text
OLTP change
   ↓
change record: order O-1057 status Packed → Shipped
   ↓
analytical destination applies the change
```

CDC can reduce repeated full scans and support fresher destinations.

Trade-offs include ordering, duplicates, deleted records, schema changes, and replay handling.


# Streaming movement

Streaming moves events continuously or with very small delay.

```text
order created ─► event channel ─► processing ─► analytical update
payment made  ─► event channel ─► processing ─► alert
```

**Advantages**

- Low-latency availability
- Supports rapid reactions

**Trade-offs**

- More moving parts
- Late and out-of-order events
- Continuous monitoring and recovery
- Harder reasoning about completeness


# Batch, incremental, or streaming?

| Requirement | Likely pattern |
|---|---|
| Monthly regulatory report | Batch |
| Daily sales dashboard | Batch or incremental |
| Inventory refreshed every 15 minutes | Incremental |
| Fraud response in seconds | Streaming |
| Rebuild five years of history | Batch |
| Live shipment status | Streaming |

Use the least complex pattern that meets the required freshness and reliability.


# Time creates subtle problems

An order can have:

- Event time: when the customer ordered
- Commit time: when OLTP saved it
- Capture time: when movement detected it
- Processing time: when the warehouse transformed it
- Availability time: when consumers could query it

```text
event → commit → capture → process → available
```

Each gap contributes to data latency.


# Data can arrive more than once

Retries are necessary in distributed systems:

```text
send order change
     ↓
acknowledgement lost
     ↓
sender retries the same change
```

The destination may receive a duplicate.

Reliable movement often needs an event identifier and **idempotent** behavior: processing the same event again should not corrupt the result.


# Data can arrive out of order

```text
Actual order:   Created → Paid → Shipped
Arrival order:  Paid → Created → Shipped
```

Possible causes:

- Network delay
- Parallel paths
- Retry timing
- Offline devices

Processing must distinguish event time from arrival time and define how long to wait for late data.


# Failure boundaries

```text
source → extract → transport → landing → transform → warehouse
```

Each arrow or box can fail independently.

Architecture should answer:

- Where is progress recorded?
- Can processing safely restart?
- Is partial output visible?
- Can lost data be replayed?
- How is failure detected?
- Who responds?

Reliable systems plan recovery before failure occurs.


# End-to-end ShopNow architecture

```text
                         OPERATIONAL
Customers ─► Web app ─► Orders OLTP ◄─► Inventory OLTP
                              │
                    incremental/CDC movement
                              ▼
                    landing and validation
                              ▼
               standardize + join + calculate
                              ▼
                  ANALYTICAL WAREHOUSE (MPP)
                    ↙          ↓          ↘
              dashboard     analyst     forecast
```

Each system remains optimized for its responsibility.


# Architectural pros and cons

| Choice | Benefit | Cost |
|---|---|---|
| Separate OLTP and OLAP | Workload isolation | Data movement and latency |
| Strong transaction isolation | Easier correctness | More blocking |
| Fine-grained locks | Higher concurrency | More coordination |
| Sharding | Distributed transaction scale | Cross-shard complexity |
| MPP warehouse | Parallel analytics | Distribution and skew concerns |
| Streaming movement | Fresh data | Operational complexity |

Architecture is the management of these trade-offs.


# Group activity — design the path

ShopNow needs:

- Checkout response below one second
- Inventory correctness during flash sales
- Sales dashboard refreshed every 30 minutes
- Five years of product history
- Analysts must not slow checkout

Draw an architecture and label:

1. Operational and analytical systems
2. Scaling choices
3. Movement pattern
4. One lock/contention risk
5. One failure-recovery requirement

**Time:** 8 minutes · **Share:** 2 minutes


# Common misconceptions

❌ “Locks are bad.”  
✅ Locks protect correctness; excessive contention is the problem.

❌ “Sharding makes every query faster.”  
✅ Cross-shard work may become harder.

❌ “A warehouse is a backup of OLTP.”  
✅ It is organized for integrated historical analysis.

❌ “MPP makes all workloads fast.”  
✅ It helps work that can be divided effectively.

❌ “Streaming is always the modern choice.”  
✅ Freshness must justify complexity.


# Knowledge check

Explain in your own words:

1. Why does an OLTP system use transactions?
2. How does lock scope affect concurrency?
3. What is the difference between blocking and deadlock?
4. How do partitioning and sharding differ?
5. Why do read replicas not directly scale writes?
6. Why is OLAP separated from OLTP?
7. How does MPP process a large query?
8. When would incremental movement be preferable to full movement?


# Architecture review checklist

When reading or designing an architecture, ask:

- What workload does each system serve?
- Where is the authoritative current state?
- What requires transaction correctness?
- Where might contention occur?
- Which scaling dimension is under pressure?
- Where is historical analytical data stored?
- How is parallel work distributed?
- How fresh must analytical data be?
- How are duplicates, late data, and failures handled?
- Which trade-off is being accepted?


# Complete mental model

```text
CURRENT BUSINESS                              BUSINESS OVER TIME

users → application → OLTP ── movement ──► warehouse/OLAP → consumers
                        │                       │
                  transactions            historical data
                  concurrency             dimensional model
                  locks                   MPP execution
                  scale pressure          large aggregations

             correctness       trust       insight
```

Operational and analytical architecture solve different problems while sharing the same business facts.


# Key takeaways

- OLTP systems safely process many short, concurrent business transactions
- Transactions and locks exchange some concurrency for correctness
- Lock scope and duration determine blocking impact
- Vertical scaling, replicas, partitioning, and sharding solve different pressures
- OLAP systems preserve and analyze integrated history
- Warehouses organize data around analytical business questions
- MPP divides large analytical work across nodes
- Data moves to isolate workloads, preserve history, and integrate meaning
- Batch, incremental, CDC, and streaming provide different freshness–complexity trade-offs


<div style="padding:2.2rem;border-radius:18px;background:linear-gradient(135deg,#3f1d68,#0f766e);color:white;text-align:center;">
<h1 style="font-size:2.8rem;">Run the present. Understand the past.</h1>
<p style="font-size:1.35rem;">Operational systems protect business transactions; analytical systems reveal patterns; data movement connects the two.</p>
<hr style="border:0;border-top:1px solid rgba(255,255,255,.4);margin:2rem;">
<h2>Questions and discussion</h2>
</div>
